# 3. BaTiO₃ finite-field MD and hysteresis-style loops

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/03_batio3_finite_field_md.ipynb)

This standalone example runs a small, one-cell ASE Langevin trajectory while cycling the field in both directions. It compares two thermostat temperatures and records polarization at each field segment. It uses the bundled model and does not require the model trained in notebook 1.

The checkpoint was trained on an analytic fixed-cell Landau teaching surface. The resulting loops show how a field schedule, temperature, and finite-rate dynamics affect a model trajectory. They are not quantitative BaTiO₃ hysteresis predictions: the model has no domain walls, nucleation, or validated DFT data along the reversal path.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Set up a two-temperature field cycle

The field amplitude stays within the toy training range of $\pm0.12$ V/Å. The trajectory is deliberately short; increase the number of points and steps only after checking runtime and stability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACEField-Landau.model"
TRAIN = DATA_DIR / "batio3/macefield_batio3_toy_train.extxyz"
source = read(TRAIN, index=0)
source_displacement = float(source.info["REF_mode_displacement"])
ti_index = source.get_chemical_symbols().index("Ti")
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)

field_up = np.linspace(-0.10, 0.10, 21)
field_cycle = np.concatenate([field_up, field_up[-2::-1]])
temperatures_K = [25.0, 200.0]
steps_per_field = 20
dt_fs = 0.5
friction_per_fs = 0.02
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

records_by_temperature = {}
for temperature_K in temperatures_K:
    atoms = source.copy()
    atoms.positions[ti_index, 2] += -0.15 - source_displacement
    rng = np.random.default_rng(23 + int(temperature_K))
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
    Stationary(atoms)

    calc = MACECalculator(
        model_paths=str(MODEL),
        model_type="MACEField",
        device=device,
        default_dtype="float32",
        electric_field=[0.0, 0.0, float(field_cycle[0])],
        compute_polarization=True,
        compute_becs=False,
        compute_polarizability=False,
    )
    atoms.calc = calc
    dyn = Langevin(
        atoms,
        timestep=dt_fs * units.fs,
        temperature_K=temperature_K,
        friction=friction_per_fs / units.fs,
        rng=rng,
    )

    polarization_z = []
    energy = []
    for field_z in field_cycle:
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        dyn.run(steps_per_field)
        energy.append(atoms.get_potential_energy())
        polarization_z.append(float(calc.results["polarization"][2]))

    records_by_temperature[temperature_K] = {
        "field_z": field_cycle.copy(),
        "polarization_z": np.asarray(polarization_z),
        "energy": np.asarray(energy),
    }
    print(
        f"{temperature_K:g} K: {len(field_cycle) * steps_per_field} MD steps, "
        f"final energy {energy[-1]:.6f} eV"
    )


## Plot polarization against field

Polarization is projected on the z axis and converted to microcoulombs per square centimetre. The trajectory is a finite-rate model response, not an equilibrium loop.


In [ ]:
P_TO_UC_CM2 = 1602.176634
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
n_up = len(field_up)
for temperature_K, data in records_by_temperature.items():
    p = data["polarization_z"] * P_TO_UC_CM2
    axes[0].plot(data["field_z"][:n_up], p[:n_up], "o-", ms=3,
                 label=f"{temperature_K:g} K, increasing")
    axes[0].plot(data["field_z"][n_up - 1:], p[n_up - 1:], "o--", ms=3,
                 label=f"{temperature_K:g} K, decreasing")
    axes[1].plot(np.arange(len(p)) * steps_per_field * dt_fs, p,
                 label=f"{temperature_K:g} K")
axes[0].set(xlabel="Ez (V/Å)", ylabel="Pz (µC/cm²)",
            title="Finite-rate polarization loop")
axes[1].set(xlabel="time (fs)", ylabel="Pz (µC/cm²)",
            title="Polarization during the cycle")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## Interpretation

The temperature comparison changes the trajectory through the same learned energy surface. A predictive hysteresis study needs reference structures and response labels at both field polarities, multiple field magnitudes, switching-path configurations, larger cells, slower sweeps, and independent validation.
